# 06 — Bias Correction and Multi-Model Ensemble Analysis
## Part 3 — Statistical Bias-Correction Methods

Part 1 established data discovery and quality control. Part 2 established model–reference diagnostics. Part 3 implements the main statistical bias-correction methods and a reproducible calibration/validation design.

Methods covered:

1. Additive / linear scaling
2. Multiplicative scaling
3. Variance scaling
4. Empirical Quantile Mapping (EQM)
5. Independent validation
6. Before/after correction diagnostics

The correction is estimated from a historical calibration period and then applied to validation or future model data without using future observations to fit the correction.


## 1. Scientific Principle

A bias-correction function is estimated from a historical model/reference relationship.

The correct sequence is:

**Historical model + reference → calibration → correction parameters**

then:

**Independent historical model → validation**

and:

**Future model → corrected projection**

Reference observations from the validation or future period must not be used to estimate the correction parameters.


## 2. Configuration

The periods below are examples. Replace them with the periods defined by your study.

Using a separate validation period is strongly recommended because a correction that reproduces its calibration period is not necessarily reliable outside that period.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from scipy import stats

warnings.filterwarnings("ignore")

CAL_START = "1981-01-01"
CAL_END = "2000-12-31"

VAL_START = "2001-01-01"
VAL_END = "2010-12-31"

VARIABLE = "tasmax"

print("Calibration:", CAL_START, "to", CAL_END)
print("Validation :", VAL_START, "to", VAL_END)
print("Variable   :", VARIABLE)


## 3. Time-Dimension Helper

In [ ]:
def get_time_dim(da):
    for name in ["time", "Time", "date", "datetime"]:
        if name in da.dims:
            return name
        if name in da.coords:
            return name
    raise ValueError("Time coordinate not found.")


## 4. Select a Period

In [ ]:
def select_period(da, start, end):
    time_dim = get_time_dim(da)
    return da.sel({time_dim: slice(start, end)})


## 5. Align Model and Reference

Only common timestamps should be used for calibration and validation comparisons. Calendar differences should be resolved before this stage.


In [ ]:
def align_model_reference(model, reference):
    model_time = get_time_dim(model)
    reference_time = get_time_dim(reference)

    if model_time != reference_time:
        reference = reference.rename({reference_time: model_time})

    return xr.align(model, reference, join="inner")


## 6. Additive / Linear Scaling

For temperature-like variables, a simple correction is:

`X_corrected = X_model + (mean_reference - mean_model)`

It removes the calibration-period mean offset while retaining the model's temporal anomalies.


In [ ]:
def fit_additive_scaling(model_cal, reference_cal):
    time_dim = get_time_dim(model_cal)

    model_mean = model_cal.mean(dim=time_dim, skipna=True)
    reference_mean = reference_cal.mean(dim=time_dim, skipna=True)

    return reference_mean - model_mean


def apply_additive_scaling(model, adjustment):
    return model + adjustment


## 7. Additive Scaling Example

In [ ]:
# Example:
#
# model_cal = select_period(model_da, CAL_START, CAL_END)
# reference_cal = select_period(reference_da, CAL_START, CAL_END)
# model_cal, reference_cal = align_model_reference(model_cal, reference_cal)
#
# adjustment = fit_additive_scaling(model_cal, reference_cal)
# corrected_cal = apply_additive_scaling(model_cal, adjustment)


## 8. Multiplicative Scaling

For precipitation amounts, a multiplicative correction is often appropriate:

`X_corrected = X_model * mean_reference / mean_model`

The method must protect against zero or near-zero model means.


In [ ]:
def fit_multiplicative_scaling(model_cal, reference_cal, minimum_mean=1e-12):
    time_dim = get_time_dim(model_cal)

    model_mean = model_cal.mean(dim=time_dim, skipna=True)
    reference_mean = reference_cal.mean(dim=time_dim, skipna=True)

    return xr.where(
        np.abs(model_mean) > minimum_mean,
        reference_mean / model_mean,
        np.nan,
    )


def apply_multiplicative_scaling(model, ratio):
    return model * ratio


## 9. Additive vs Multiplicative Scaling

| Variable | Typical first candidate |
|---|---|
| Temperature | Additive scaling |
| Precipitation amount | Multiplicative scaling |
| Precipitation occurrence | Separate occurrence treatment may be required |
| Climate indicators | Depends on physical definition |

These are starting points, not universal rules. Validation performance should determine the final method.


## 10. Variance Scaling

Variance scaling corrects both the mean and standard deviation:

`X_corrected = mean_reference + (X_model - mean_model) * std_reference / std_model`

This is useful when the model has a systematic variance error in addition to a mean bias.


In [ ]:
def fit_variance_scaling(model_cal, reference_cal, minimum_std=1e-12):
    time_dim = get_time_dim(model_cal)

    model_mean = model_cal.mean(dim=time_dim, skipna=True)
    reference_mean = reference_cal.mean(dim=time_dim, skipna=True)
    model_std = model_cal.std(dim=time_dim, skipna=True)
    reference_std = reference_cal.std(dim=time_dim, skipna=True)

    scale = xr.where(
        model_std > minimum_std,
        reference_std / model_std,
        np.nan,
    )

    return {
        "model_mean": model_mean,
        "reference_mean": reference_mean,
        "scale": scale,
    }


def apply_variance_scaling(model, parameters):
    return (
        parameters["reference_mean"]
        + (model - parameters["model_mean"]) * parameters["scale"]
    )


## 11. Empirical Quantile Mapping (EQM)

EQM corrects the complete empirical distribution.

For each model value, its percentile in the model calibration distribution is estimated and the corresponding percentile in the reference calibration distribution is used as the corrected value.

EQM can modify mean, variance, skewness, and distribution tails.


In [ ]:
def empirical_quantile_mapping_1d(
    model_cal,
    reference_cal,
    values,
    n_quantiles=1001,
):
    model_cal = np.asarray(model_cal, dtype=float)
    reference_cal = np.asarray(reference_cal, dtype=float)
    values = np.asarray(values, dtype=float)

    model_cal = model_cal[np.isfinite(model_cal)]
    reference_cal = reference_cal[np.isfinite(reference_cal)]

    result = np.full_like(values, np.nan, dtype=float)

    if model_cal.size < 2 or reference_cal.size < 2:
        return result

    probabilities = np.linspace(0, 1, n_quantiles)
    model_q = np.quantile(model_cal, probabilities)
    reference_q = np.quantile(reference_cal, probabilities)

    unique_model, unique_index = np.unique(model_q, return_index=True)
    unique_reference = reference_q[unique_index]

    valid = np.isfinite(values)

    if unique_model.size == 1:
        result[valid] = unique_reference[0]
    else:
        result[valid] = np.interp(
            values[valid],
            unique_model,
            unique_reference,
            left=unique_reference[0],
            right=unique_reference[-1],
        )

    return result


## 12. EQM on Gridded xarray Data

For gridded data, EQM should normally be estimated separately at each grid cell after the model and reference grids have been made compatible.

The wrapper below applies the one-dimensional mapping along time.


In [ ]:
def eqm_correct(model_cal, reference_cal, values, n_quantiles=1001):
    model_time = get_time_dim(model_cal)
    reference_time = get_time_dim(reference_cal)
    values_time = get_time_dim(values)

    if model_time != "time":
        model_cal = model_cal.rename({model_time: "time"})
    if reference_time != "time":
        reference_cal = reference_cal.rename({reference_time: "time"})
    if values_time != "time":
        values = values.rename({values_time: "time"})

    corrected = xr.apply_ufunc(
        empirical_quantile_mapping_1d,
        model_cal,
        reference_cal,
        values,
        input_core_dims=[["time"], ["time"], ["time"]],
        output_core_dims=[["time"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        kwargs={"n_quantiles": n_quantiles},
    )

    if values_time != "time":
        corrected = corrected.rename({"time": values_time})

    return corrected


## 13. Calibration and Validation Data

Separate the historical period before fitting any correction.

The same fitted parameters are then applied to validation data. This prevents optimistic validation caused by fitting and evaluating on the same observations.


In [ ]:
# Example:
#
# model_cal = select_period(model_da, CAL_START, CAL_END)
# ref_cal = select_period(reference_da, CAL_START, CAL_END)
# model_val = select_period(model_da, VAL_START, VAL_END)
# ref_val = select_period(reference_da, VAL_START, VAL_END)
#
# model_cal, ref_cal = align_model_reference(model_cal, ref_cal)
# model_val, ref_val = align_model_reference(model_val, ref_val)


## 14. Apply Additive Correction to Validation

In [ ]:
# Example:
#
# adjustment = fit_additive_scaling(model_cal, ref_cal)
# corrected_val = apply_additive_scaling(model_val, adjustment)


## 15. Apply Multiplicative Correction to Validation

In [ ]:
# Example:
#
# ratio = fit_multiplicative_scaling(model_cal, ref_cal)
# corrected_val = apply_multiplicative_scaling(model_val, ratio)


## 16. Apply Variance Scaling to Validation

In [ ]:
# Example:
#
# parameters = fit_variance_scaling(model_cal, ref_cal)
# corrected_val = apply_variance_scaling(model_val, parameters)


## 17. Apply EQM to Validation

The calibration distributions remain fixed. Validation model values are mapped through the historical model/reference quantile relationship.


In [ ]:
# Example:
#
# corrected_val = eqm_correct(
#     model_cal,
#     ref_cal,
#     model_val,
#     n_quantiles=1001,
# )


## 18. Evaluation Metrics

Use multiple statistics:

- mean bias
- MAE
- RMSE
- standard-deviation ratio
- correlation
- selected quantile errors

Improvement in one metric does not prove that the corrected dataset is universally better.


In [ ]:
def metric_summary(model_da, reference_da):
    time_dim = get_time_dim(model_da)

    error = model_da - reference_da
    bias = error.mean(dim=time_dim, skipna=True)
    mae = np.abs(error).mean(dim=time_dim, skipna=True)
    rmse = np.sqrt((error ** 2).mean(dim=time_dim, skipna=True))

    model_std = model_da.std(dim=time_dim, skipna=True)
    reference_std = reference_da.std(dim=time_dim, skipna=True)
    std_ratio = model_std / reference_std

    model_anom = model_da - model_da.mean(dim=time_dim, skipna=True)
    reference_anom = reference_da - reference_da.mean(
        dim=time_dim, skipna=True
    )

    numerator = (model_anom * reference_anom).sum(
        dim=time_dim, skipna=True
    )
    denominator = np.sqrt(
        (model_anom ** 2).sum(dim=time_dim, skipna=True)
        * (reference_anom ** 2).sum(dim=time_dim, skipna=True)
    )

    return {
        "bias": bias,
        "mae": mae,
        "rmse": rmse,
        "std_ratio": std_ratio,
        "correlation": numerator / denominator,
    }


## 19. Spatial Averaging of Metrics

In [ ]:
def spatial_average(da):
    spatial_names = {"lat", "latitude", "lon", "longitude", "x", "y"}
    dims = [d for d in da.dims if d.lower() in spatial_names]

    if dims:
        return da.mean(dim=dims, skipna=True)

    return da


def metrics_to_table(metrics):
    rows = []

    for name, value in metrics.items():
        value = spatial_average(value)
        try:
            value = float(value.mean())
        except Exception:
            value = np.nan

        rows.append({"metric": name, "value": value})

    return pd.DataFrame(rows)


## 20. Raw vs Corrected Comparison

The key validation comparison is:

**raw model → reference**

versus

**corrected model → reference**

Use the independent validation period for this final performance assessment.


In [ ]:
def compare_raw_and_corrected(raw_model, corrected_model, reference):
    raw_metrics = metric_summary(raw_model, reference)
    corrected_metrics = metric_summary(corrected_model, reference)

    rows = []

    for metric in raw_metrics:
        raw_value = spatial_average(raw_metrics[metric])
        corrected_value = spatial_average(corrected_metrics[metric])

        rows.append({
            "metric": metric,
            "raw": float(raw_value.mean()),
            "corrected": float(corrected_value.mean()),
        })

    return pd.DataFrame(rows)


## 21. Empirical CDF Diagnostic

Distribution-based methods should be evaluated using empirical cumulative distribution functions.

The corrected distribution should approach the reference distribution across both the central distribution and, where appropriate, the tails.


In [ ]:
def plot_ecdf_comparison(
    raw_model,
    reference,
    corrected=None,
    sample_size=20000,
    seed=42,
):
    arrays = [raw_model, reference]
    labels = ["Raw model", "Reference"]

    if corrected is not None:
        arrays.append(corrected)
        labels.append("Corrected")

    rng = np.random.default_rng(seed)

    plt.figure(figsize=(10, 6))

    for da, label in zip(arrays, labels):
        values = np.asarray(da.values).ravel()
        values = values[np.isfinite(values)]

        if values.size > sample_size:
            values = rng.choice(values, sample_size, replace=False)

        values = np.sort(values)
        probability = np.arange(1, values.size + 1) / values.size

        plt.plot(values, probability, label=label)

    plt.xlabel("Value")
    plt.ylabel("Empirical cumulative probability")
    plt.title("Empirical CDF Comparison")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 22. Quantile–Quantile Diagnostic

In [ ]:
def plot_qq_comparison(
    raw_model,
    reference,
    corrected=None,
    n_quantiles=99,
):
    q = np.linspace(0.01, 0.99, n_quantiles)

    ref = np.asarray(reference.values).ravel()
    ref = ref[np.isfinite(ref)]

    raw = np.asarray(raw_model.values).ravel()
    raw = raw[np.isfinite(raw)]

    ref_q = np.quantile(ref, q)
    raw_q = np.quantile(raw, q)

    plt.figure(figsize=(7, 7))
    plt.plot(ref_q, raw_q, label="Raw model")

    if corrected is not None:
        corr = np.asarray(corrected.values).ravel()
        corr = corr[np.isfinite(corr)]
        corr_q = np.quantile(corr, q)

        plt.plot(ref_q, corr_q, label="Corrected")

    lower = ref_q.min()
    upper = ref_q.max()

    plt.plot(
        [lower, upper],
        [lower, upper],
        linestyle="--",
        label="1:1",
    )

    plt.xlabel("Reference quantile")
    plt.ylabel("Model quantile")
    plt.title("Quantile–Quantile Comparison")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 23. Physical Plausibility Check

Statistical improvement is not sufficient.

Check corrected output for:

- negative precipitation
- impossible temperature values
- unexpected NaNs
- excessive amplification of extremes
- unit consistency

For precipitation, negative values must never be accepted as physical output.


In [ ]:
def physical_range_check(da, variable):
    print(f"Variable: {variable}")
    print(f"Minimum: {float(da.min(skipna=True))}")
    print(f"Maximum: {float(da.max(skipna=True))}")
    print(f"NaN count: {int(da.isnull().sum())}")

    if variable.lower() in {"pr", "precip", "precipitation"}:
        negative = int((da < 0).sum())
        print(f"Negative values: {negative}")


## 24. Tail and Extrapolation Caution

Standard empirical quantile mapping is bounded by the calibration distribution.

Future model values may exceed the historical calibration range. A production workflow must therefore document its extrapolation rule.

The simple EQM implementation here uses endpoint values for out-of-range observations. For extreme-event research, tail behaviour should be tested separately.


## 25. Climate-Change Signal

A useful projection diagnostic is the mean change signal:

`delta X = mean(future) - mean(historical)`

Compare this signal before and after correction.

A correction that substantially changes the projected signal should be investigated rather than automatically accepted.


In [ ]:
def change_signal(historical, future):
    """Return future minus historical mean."""
    historical_time = get_time_dim(historical)
    future_time = get_time_dim(future)

    return (
        future.mean(dim=future_time, skipna=True)
        - historical.mean(dim=historical_time, skipna=True)
    )


## 26. Metadata for Corrected Products

Corrected files should retain enough metadata to be reproducible.

Record:

- correction method
- reference dataset
- calibration period
- variable
- model
- scenario

Never overwrite the raw model NetCDF file.


In [ ]:
def add_correction_metadata(
    da,
    method,
    calibration_start,
    calibration_end,
    reference_name,
):
    da = da.copy()

    da.attrs["bias_correction_method"] = method
    da.attrs["bias_correction_calibration_start"] = calibration_start
    da.attrs["bias_correction_calibration_end"] = calibration_end
    da.attrs["bias_correction_reference"] = reference_name

    return da


## 27. Recommended Output Structure

Keep corrected products separate from raw data:

```text
outputs/
└── bias_correction/
    ├── calibration/
    ├── validation/
    ├── future/
    ├── diagnostics/
    └── metrics/
```

Use filenames that identify model, variable, scenario, period, and correction method.


## 28. End-to-End Example Skeleton

In [ ]:
# ---------------------------------------------------------------
# EXAMPLE WORKFLOW
# ---------------------------------------------------------------

# model = xr.open_dataset("historical_model.nc")[VARIABLE]
# reference = xr.open_dataset("reference.nc")[VARIABLE]

# Calibration
# model_cal = select_period(model, CAL_START, CAL_END)
# ref_cal = select_period(reference, CAL_START, CAL_END)
# model_cal, ref_cal = align_model_reference(model_cal, ref_cal)

# Validation
# model_val = select_period(model, VAL_START, VAL_END)
# ref_val = select_period(reference, VAL_START, VAL_END)
# model_val, ref_val = align_model_reference(model_val, ref_val)

# Fit
# adjustment = fit_additive_scaling(model_cal, ref_cal)

# Apply
# corrected_val = apply_additive_scaling(model_val, adjustment)

# Evaluate
# results = compare_raw_and_corrected(
#     model_val,
#     corrected_val,
#     ref_val,
# )
# display(results)


## 29. Method Selection Strategy

A practical research workflow is:

1. Test a simple method first.
2. Evaluate calibration performance.
3. Evaluate independent validation performance.
4. Examine the seasonal cycle.
5. Examine the complete distribution.
6. Inspect extremes.
7. Check physical plausibility.
8. Compare preservation of the climate-change signal.

Do not select EQM simply because it produces the smallest historical error. Its treatment of extremes, temporal dependence, spatial dependence, and future extrapolation must also be considered.


## 30. Reproducibility Checklist

Before scaling the correction to all models:

- [ ] calibration period documented
- [ ] validation period documented
- [ ] model/reference grids compatible
- [ ] units verified
- [ ] calendars handled
- [ ] missing values checked
- [ ] correction method selected
- [ ] correction parameters reproducibly defined
- [ ] validation metrics calculated
- [ ] physical plausibility verified
- [ ] climate-change signal examined
- [ ] raw files preserved


## 31. Part 3 Summary

Part 3 implemented the core correction toolkit:

- additive scaling
- multiplicative scaling
- variance scaling
- empirical quantile mapping
- independent calibration/validation
- bias, MAE, RMSE, standard-deviation ratio, and correlation
- ECDF and Q–Q diagnostics
- physical plausibility checks
- metadata traceability
- climate-signal preservation checks

### Next: Part 4

Part 4 will scale the validated correction framework to the complete multi-model ensemble, including automatic model discovery, model-by-model processing, ensemble mean/median, inter-model spread, percentiles, model agreement, uncertainty visualization, and ensemble NetCDF outputs.
